In [ ]:

import os
import re
import math
import json
from typing import List, Iterable, Optional

import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

try:
    import nltk
    from nltk.tokenize import sent_tokenize
except Exception:
    def sent_tokenize(text):
        return re.split(r'(?<=[.!?])\s+', text)

# ====== 설정 - 환경에 맞게 수정 ======
PATH = "D:/development/python/MovieReview/machine-learning-team-project/data/imdb_preprocessed.csv"
TEXT_COL_HINT = "review"      # 리뷰 텍스트가 담긴 칼럼명 (없으면 자동탐지)
SAMPLE_DISPLAY_N = 20         # 출력할 샘플 행 수
OUTDIR = "./linguistic_out_vader_sample"
SAMPLE_SAVE_N = 200           # 저장할 샘플 크기
LIMIT_ROWS = 5000             # 분석할 최대 행 수
MAX_DISPLAY_LEN = 100         # 출력 리뷰 텍스트 최대 길이

os.makedirs(OUTDIR, exist_ok=True)

# VADER 분석기 초기화
analyzer = SentimentIntensityAnalyzer()

# 1) 이중부정 패턴 목록
DOUBLE_NEGATION_PATTERNS = [
    r"\bnot\s+(?:un|in|im|ir|il|non|dis)[a-zA-Z']+",
    r"\bnot\s+bad\b", r"\bnot\s+un\b", r"\bcan't\s+not\b", r"\bcannot\s+not\b",
    r"\bnever\s+not\b", r"\bno\s+less\b", r"\bnot\s+so\s+bad\b", r"\bnot\s+too\s+bad\b",
    r"\bnot\s+without\b", r"\bnot\s+disappoint(ing|ed)?\b",
]

# 2) 반어(사카즘) 마커 패턴 목록
SARCASM_MARKERS = [
    r"\byeah right\b", r"\bas if\b", r"\bi love (how|that)\b", r"\bi just love\b",
    r"\bwhat a (?:treat|joy|surprise|piece)\b", r"\bbig surprise\b",
    r"\"(?:great|amazing|awesome|best)\"", r"'(?:great|amazing|awesome|best)'",
    r"\bi'm (?:so|totally|absolutely) (?:sure|happy)\b", r"\bright idea\b",
    r"\bjust great\b", r"\bjust perfect\b", r"\b(sure|right)\.\.\.\b",
]

# -----------------------
# 핵심 탐지 함수
# -----------------------
def detect_double_negation(text: str) -> List[str]:
    patterns = []
    if not text:
        return patterns
    for p in DOUBLE_NEGATION_PATTERNS:
        if re.search(p, text, re.IGNORECASE):
            patterns.append(p)
    return patterns

def detect_sarcasm_candidate(text: str) -> List[str]:
    if not text:
        return []
    txt = text.strip()
    patterns = []
    for p in SARCASM_MARKERS:
        if re.search(p, txt, re.IGNORECASE):
            patterns.append(p)

    vs = analyzer.polarity_scores(txt)
    compound = vs.get('compound', 0.0)

    # 문장 분리 (fallback 포함)
    try:
        sents = sent_tokenize(txt)
    except Exception:
        sents = re.split(r'[.!?]+', txt)

    pos_sent_count = sum(1 for s in sents if analyzer.polarity_scores(s)['compound'] >= 0.5)
    neg_sent_count = sum(1 for s in sents if analyzer.polarity_scores(s)['compound'] <= -0.5)

    is_sarcastic_by_heuristic = (compound < -0.2 and pos_sent_count > 0) or \
                                (compound > 0.2 and neg_sent_count > 0)

    if is_sarcastic_by_heuristic and 'VADER_HEURISTIC_MATCH' not in patterns:
        patterns.append('VADER_HEURISTIC_MATCH')

    return patterns

# -----------------------
# 리스트 -> 문자열 변환 (견고하게 모든 컬럼 처리)
# -----------------------
def stringify_list_columns(df: pd.DataFrame, cols: Optional[Iterable[str]] = None) -> pd.DataFrame:
    df_copy = df.copy()

    # 먼저 결측값을 빈 문자열로 채움(나중에 str로 변환 시 'nan' 방지)
    df_copy = df_copy.fillna('')

    # 처리할 칼럼 목록 결정
    cols_to_process = list(df_copy.columns) if cols is None else [c for c in cols if c in df_copy.columns]

    # 지정 컬럼 우선 처리
    for c in cols_to_process:
        df_copy[c] = df_copy[c].apply(
            lambda v: ", ".join(v) if isinstance(v, (list, tuple)) and len(v) > 0 else ("" if v == '' else v)
        )

    # 안전장치: 나머지 컬럼에도 혹시 list/tuple이 남아있다면 처리
    for c in df_copy.columns:
        df_copy[c] = df_copy[c].apply(
            lambda v: ", ".join(v) if isinstance(v, (list, tuple)) else v
        )

    # 최종적으로 모든 값을 문자열로 바꿔 CSV 저장/중복 비교 안전성 확보
    # (빈 문자열은 그대로 유지)
    df_copy = df_copy.astype(str).replace('nan', '', regex=False)

    return df_copy

# -----------------------
# 출력 보조 함수: 정리 및 가독성 향상
# -----------------------
def _clean_pattern_string(s: str) -> str:
    if not isinstance(s, str):
        s = str(s)
    # 여러 치환을 순차 적용
    replacements = [
        (r'\\b', ''),         # 단어 경계 토큰 제거
        (r'\\s\+', ' '),      # \s+ -> 공백으로 표시
        (r'\(\?:', '('),      # (?: -> (
        (r'\\\(', '('),       # \( -> (
        (r'\\\)', ')'),       # \) -> )
        (r'\[\^?a-zA-Z\\\'\]\+', '+'),  # ...+ 표시 축약 (간단 가독화)
        (r'VADER_HEURISTIC_MATCH', 'VADER_HEURISTIC'),
    ]
    out = s
    for pat, repl in replacements:
        try:
            out = re.sub(pat, repl, out)
        except re.error:
            # 안전장치: 치환 실패 시 원본 유지
            pass
    # 불필요한 여러 공백 정리
    out = re.sub(r'\s{2,}', ' ', out).strip()
    return out

def clean_and_display_dataframe(df_to_display: pd.DataFrame, text_col: str, max_review_len: int):
    """
    DataFrame을 출력하기 전에 가독성을 높입니다:
    1. 리뷰 텍스트(text_col)의 길이를 지정된 길이로 자릅니다.
    2. 매칭 패턴 컬럼의 정규 표현식 기호를 간단히 정리하여 출력합니다.
    """
    df_copy = df_to_display.copy()

    # 1. 리뷰 텍스트 길이 제한
    def truncate_review(text):
        try:
            text = str(text)
        except Exception:
            return ""
        return text[:max_review_len] + '...' if len(text) > max_review_len else text

    if text_col in df_copy.columns:
        df_copy[text_col] = df_copy[text_col].apply(truncate_review)

    # 2. 매칭 패턴 정리 (출력 전용)
    pattern_cols = [c for c in df_copy.columns if 'matches' in c]
    for col in pattern_cols:
        if col in df_copy.columns:
            df_copy[col] = df_copy[col].astype(str).apply(_clean_pattern_string)

    # drop_duplicates 수행 전에 문자열로 안전 변환(이미 stringify_list_columns로 처리되었다면 중복 변환되지 않음)
    # 하지만 여기서는 표시 목적이므로 그대로 drop_duplicates 호출
    return df_copy.drop_duplicates(ignore_index=True)

# -----------------------
# 전체 DataFrame에 적용하여 후보 추출
# -----------------------
def extract_candidates(df: pd.DataFrame, text_col: str):
    """DataFrame 전체에 적용하여 후보 추출"""
    df = df.reset_index().rename(columns={"index": "_orig_index"})
    df["_text"] = df[text_col].astype(str).fillna("")

    # 후보 탐지 로직 적용
    df['double_negative_matches'] = df["_text"].apply(detect_double_negation)
    df['sarcasm_matches'] = df["_text"].apply(detect_sarcasm_candidate)

    # 필터링
    dn_df = df[df['double_negative_matches'].map(len) > 0].copy()
    sc_df = df[df['sarcasm_matches'].map(len) > 0].copy()

    return dn_df, sc_df

# -----------------------
# 메인 실행 블록
# -----------------------
def main():
    try:
        df = pd.read_csv(PATH, nrows=LIMIT_ROWS)
        print(f"데이터 로드 성공: {PATH} -> {len(df)}개 행 로드")
    except FileNotFoundError:
        raise FileNotFoundError(f"오류: 지정된 경로에 파일이 없습니다. PATH를 확인하세요: {PATH}")

    # 텍스트 칼럼 탐지
    if TEXT_COL_HINT in df.columns:
        text_col = TEXT_COL_HINT
    else:
        obj_cols = [c for c in df.columns if df[c].dtype == "object"]
        if not obj_cols:
            raise RuntimeError("텍스트 칼럼을 자동으로 찾을 수 없습니다.")
        avg_len = {c: df[c].dropna().astype(str).map(len).mean() for c in obj_cols}
        text_col = max(avg_len, key=avg_len.get)
    print("텍스트 칼럼 사용:", text_col)

    # 후보 추출
    dn_df_vader, sc_df_vader = extract_candidates(df, text_col=text_col)

    # 결과 요약
    print("\n\n=== VADER 기반 매칭 요약 ===")
    print(f"총 분석 리뷰: {len(df):,}")
    print(f"이중부정 후보: {len(dn_df_vader):,}개")
    print(f"반어/풍자 후보 (VADER 휴리스틱 포함): {len(sc_df_vader):,}개")
    print("----------------------------------")

    # 출력용 칼럼 목록
    display_cols_base = ['_orig_index', text_col, 'sentiment', 'double_negative_matches', 'sarcasm_matches']
    display_cols = [c for c in display_cols_base if c in dn_df_vader.columns or c == text_col]

    # 1) 이중부정 후보 출력
    dn_display = stringify_list_columns(dn_df_vader[display_cols], ["double_negative_matches", "sarcasm_matches"])
    final_dn_display = clean_and_display_dataframe(dn_display, text_col, max_review_len=MAX_DISPLAY_LEN)

    print(f"\n\n--- 이중부정 후보 샘플 (Top {SAMPLE_DISPLAY_N}) ---")
    if 'display' in globals():
        display(final_dn_display.head(SAMPLE_DISPLAY_N))
    else:
        print(final_dn_display.head(SAMPLE_DISPLAY_N).to_string(index=False))

    # 2) 반어/풍자 후보 출력
    sc_display = stringify_list_columns(sc_df_vader[display_cols], ["double_negative_matches", "sarcasm_matches"])
    final_sc_display = clean_and_display_dataframe(sc_display, text_col, max_review_len=MAX_DISPLAY_LEN)

    print(f"\n\n--- 반어/풍자 후보 샘플 (Top {SAMPLE_DISPLAY_N}) ---")
    if 'display' in globals():
        display(final_sc_display.head(SAMPLE_DISPLAY_N))
    else:
        print(final_sc_display.head(SAMPLE_DISPLAY_N).to_string(index=False))

    # 3) CSV 저장 (리스트 -> 문자열 변환 후 안전하게 저장)
    save_cols = [c for c in dn_df_vader.columns if c != '_text']

    # 디버그: 리스트형 컬럼이 남아있는지 확인
    list_like_cols_dn = [c for c in dn_df_vader.columns if dn_df_vader[c].apply(lambda x: isinstance(x, (list, tuple))).any()]
    list_like_cols_sc = [c for c in sc_df_vader.columns if sc_df_vader[c].apply(lambda x: isinstance(x, (list, tuple))).any()]
    print("\n[디버그] 이중부정 데이터프레임에 list/tuple 타입이 남아있는 컬럼:", list_like_cols_dn)
    print("[디버그] 반어/풍자 데이터프레임에 list/tuple 타입이 남아있는 컬럼:", list_like_cols_sc)

    # 이중부정 후보 저장
    dn_save = stringify_list_columns(dn_df_vader[save_cols], ["double_negative_matches", "sarcasm_matches"])
    dn_save = dn_save.drop_duplicates(ignore_index=True)
    dn_sample_df = dn_save.sample(n=min(SAMPLE_SAVE_N, len(dn_save)), random_state=42) if len(dn_save) > 0 else dn_save
    dn_out_path = os.path.join(OUTDIR, f"vader_double_negative_sample_{len(dn_sample_df)}.csv")
    dn_sample_df.to_csv(dn_out_path, index=False)
    print(f"\n이중부정 후보 샘플 CSV 파일 저장 완료: {dn_out_path}")

    # 반어/풍자 후보 저장
    sc_save = stringify_list_columns(sc_df_vader[save_cols], ["double_negative_matches", "sarcasm_matches"])
    sc_save = sc_save.drop_duplicates(ignore_index=True)
    sc_sample_df = sc_save.sample(n=min(SAMPLE_SAVE_N, len(sc_save)), random_state=42) if len(sc_save) > 0 else sc_save
    sc_out_path = os.path.join(OUTDIR, f"vader_sarcasm_sample_{len(sc_sample_df)}.csv")
    sc_sample_df.to_csv(sc_out_path, index=False)
    print(f"반어/풍자 후보 샘플 CSV 파일 저장 완료: {sc_out_path}")

if __name__ == "__main__":
    main()


데이터 로드 성공: D:/development/python/MovieReview/machine-learning-team-project/data/imdb_preprocessed.csv -> 5000개 행 로드
텍스트 칼럼 사용: review


=== VADER 기반 매칭 요약 ===
총 분석 리뷰: 5,000
이중부정 후보: 184개
반어/풍자 후보 (VADER 휴리스틱 포함): 213개
----------------------------------


--- 이중부정 후보 샘플 (Top 20) ---
_orig_index                                                                                                  review sentiment                                         double_negative_matches sarcasm_matches
          8 encouraged by the positive comments about this film on here i was looking forward to watching this f...  negative                                                         no less                
         14 this a fantastic movie of three prisoners who become famous one of the actors is george clooney and ...  positive                                                         not bad                
         21 i had the terrible misfortune of having to view this b movie in its entirety all i hav